### Implementação de Fuzzy C-means utilizando GloVe

In [2]:
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
import plotly.express as px
import skfuzzy as fuzz
from plotly.subplots import make_subplots


In [3]:
def load_glove(file_path):
    embeddings = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            embeddings[word] = vector
    return embeddings

def words_to_vectors(words, embeddings, dimension=50):
    vectors = []
    for word in words:
        vector = embeddings.get(word)
        if vector is not None:
            vectors.append(vector)
        else:
            print(f"'{word}' not found in GloVe vocabulary. Using zero vector.")
            vectors.append(np.zeros(dimension))
    return np.array(vectors)

glove_file_path = './GloVe/glove.6B.50d.txt'

glove_embeddings = load_glove(glove_file_path)


In [4]:
words = ['harvard', 'learning', 'intelligence']

word_vectors = words_to_vectors(words, glove_embeddings)
print(word_vectors)

[[-8.5970e-01  1.1120e+00 -2.9970e-01 -1.1093e+00  1.5653e-01 -1.3244e-01
  -1.0520e+00 -9.2620e-01 -5.2920e-01 -2.4501e-01 -2.2653e-01  2.5299e-01
  -9.9125e-02 -4.0640e-01  9.7853e-04 -3.5808e-02 -1.8689e-01  7.1157e-01
  -4.4480e-01  8.6651e-01  5.4339e-01  5.9826e-01 -3.1584e-02 -4.6351e-01
  -8.5038e-02 -1.8902e+00  1.1140e-01 -7.5604e-01 -1.6965e+00 -3.9752e-01
   1.2976e+00 -3.4127e-01 -2.2890e-01 -1.4524e+00 -2.9855e-01 -2.0297e-01
  -4.4211e-01  1.1521e+00  1.5059e+00 -4.8819e-01 -2.1176e-01 -3.6186e-01
  -9.1108e-02  9.5266e-01  2.0254e-01  1.0068e-01  6.9316e-01  2.6215e-01
  -9.0986e-01  5.9507e-01]
 [ 2.0461e-01  4.8659e-01 -5.5308e-01 -2.7019e-01  2.6336e-01  1.5751e-01
  -2.8994e-01 -5.1824e-01  5.1829e-02  3.6225e-01  3.7077e-01  1.3220e-01
  -6.1377e-02 -5.3606e-01 -3.4733e-01 -4.3981e-02 -8.6744e-02  7.8305e-01
   4.1422e-01  2.7996e-02  2.3433e-01  9.8844e-01 -4.1049e-01  6.2060e-01
   1.3966e+00 -6.5427e-01 -1.8221e-01 -1.0293e+00 -1.4741e-02 -2.5384e-01
   3.2270e+

In [5]:
wordsim_path_file = './WordSim_353/wordsim_relatedness_goldstandard.txt'
df = pd.read_csv(wordsim_path_file, sep='\t', header=None)
print(df.head())
words = pd.concat([df[0], df[1]]).str.lower().drop_duplicates()
word_vectors = words_to_vectors(words, glove_embeddings)
print('#words = ', len(words))

           0          1     2
0   computer   keyboard  7.62
1  Jerusalem     Israel  8.46
2     planet     galaxy  8.11
3     canyon  landscape  7.53
4       OPEC    country  5.63
#words =  346


In [6]:
# perplexity = np.arange(10, 300, 10)
# divergence = []

# for i in perplexity:
#     model = TSNE(n_components=2, init="pca", perplexity=i)
#     reduced = model.fit_transform(word_vectors)
#     divergence.append(model.kl_divergence_)
# fig = px.line(x=perplexity, y=divergence, markers=True)
# fig.update_layout(xaxis_title="Perplexity Values", yaxis_title="Divergence")
# fig.update_traces(line_color="red", line_width=1)
# fig.show()

In [7]:
tsne = TSNE(n_components=2,perplexity=20, init='pca', random_state=0)
word_vectors_tsne = tsne.fit_transform(word_vectors)

tsne.kl_divergence_

1.2116248607635498

In [8]:
fig = px.scatter(x=word_vectors_tsne[:, 0], y=word_vectors_tsne[:, 1], text=words)
fig.update_layout(
    title="t-SNE visualization of WordSim_353 dataset",
    xaxis_title="First t-SNE",
    yaxis_title="Second t-SNE",
    width=800,
    height=500
)
fig.show()

In [16]:
n_clusters = 2
fuzz_par = 1.1

fcm_model = fuzz.cluster.cmeans(word_vectors.T, c=n_clusters, m=fuzz_par, error=0.05, maxiter=1000, init=None)
cluster_membership = np.argmax(fcm_model[1], axis=0)

df_membership = pd.DataFrame(fcm_model[1].T, index=words, columns=[f"Cluster {i}" for i in range(n_clusters)])

top_words_per_cluster = {}

top_n = 100

for i in range(n_clusters):
    # Seleciona as top_n palavras ordenadas pelo grau de pertencimento ao cluster i
    top_entries = df_membership[f"Cluster {i}"].sort_values(ascending=False).head(top_n)
    top_words_per_cluster[f"Cluster {i}"] = [f"{word} ({value:.4f})" for word, value in zip(top_entries.index, top_entries.values)]

df_top_words = pd.DataFrame.from_dict(top_words_per_cluster, orient="index")

display(df_top_words)

df_membership = pd.DataFrame(fcm_model[1].T, index=words, columns=[f"Cluster {i}" for i in range(n_clusters)])

df_membership = df_membership.map(lambda x: f"{x:.4f}")

display(df_membership)

fcm_im = fcm_model[1] @ fcm_model[1].T

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
Cluster 0,magician (0.9960),wizard (0.9943),lover (0.9929),exhibit (0.9922),graveyard (0.9914),monk (0.9875),closet (0.9859),lad (0.9847),virtuoso (0.9816),freud (0.9767),...,holy (0.8583),inmate (0.8580),racket (0.8562),combination (0.8540),seepage (0.8540),tennis (0.8521),sex (0.8511),concert (0.8487),family (0.8462),hypertension (0.8445)
Cluster 1,month (0.9990),government (0.9982),plan (0.9975),possibility (0.9974),issue (0.9966),country (0.9959),wednesday (0.9946),year (0.9944),announcement (0.9944),change (0.9939),...,governor (0.8837),war (0.8821),fear (0.8818),ticket (0.8798),benchmark (0.8720),chance (0.8681),law (0.8667),arafat (0.8626),line (0.8575),registration (0.8569)


,Cluster 0,Cluster 1
computer,0.5741,0.4259
jerusalem,0.3388,0.6612
planet,0.9388,0.0612
canyon,0.8640,0.1360
opec,0.0998,0.9002
...,...,...
voyage,0.7992,0.2008
string,0.6671,0.3329
smile,0.9587,0.0413
cucumber,0.8951,0.1049


In [19]:
fig = px.scatter(x=word_vectors_tsne[:, 0], y=word_vectors_tsne[:, 1], text=words, color=cluster_membership)
fig.update_layout(
    title="t-SNE visualization of WordSim_353 dataset with FCM",
    xaxis_title="First t-SNE",
    yaxis_title="Second t-SNE",
    width=1500,
    height=900
)
fig.show()

fcm_im = fcm_model[1].T @ fcm_model[1]

df_membership.index = np.array(df_membership.index)
df_membership['cluster'] = cluster_membership

df_membership_ord = df_membership.sort_values(by='cluster')
df_membership_ord = df_membership_ord.drop(columns='cluster')
# display(df_membership_ord)

df_membership_ord = np.array(df_membership_ord).astype(float)
fcm_im_ord = df_membership_ord @ df_membership_ord.T

# Create a subplot with two columns
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Affinity matrix", "Ordered affinity matrix"),
    column_widths=[0.5, 0.5]
)

# Add the first heatmap
fig.add_trace(
    px.imshow(fcm_im, color_continuous_scale='plasma').data[0],
    row=1, col=1
)

# Add the second heatmap
fig.add_trace(
    px.imshow(fcm_im_ord, color_continuous_scale='plasma').data[0],
    row=1, col=2
)

# Update layout
fig.update_layout(
    title="Comparison of FCM Membership Matrices",
    width=1200,
    height=600
)

fig.show()